# Task 1

**installation environment**

In [ ]:
!pip install transformers datasets accelerate scikit-learn

**Download the dataset**

In [ ]:
# Download the main dataset file
!wget -O dontpatronizeme_pcl.tsv https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv

# Download the partition files of the training set and the development set
!wget https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv
!wget https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv

# Download the test set
!wget https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv

# Download the classification details dataset
!wget -O dontpatronizeme_categories.tsv https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_categories.tsv

**Construct the official split**

In [ ]:
import pandas as pd

data = pd.read_csv("dontpatronizeme_pcl.tsv", sep="\t", skiprows=4)
data.columns = ["par_id","art_id","keyword","country","text","orig_label"]

# turn to binary
data["label"] = data["orig_label"].apply(lambda x: 0 if x in [0,1] else 1)

train_ids = pd.read_csv("train_semeval_parids-labels.csv")
dev_ids = pd.read_csv("dev_semeval_parids-labels.csv")

data['par_id_str'] = data['par_id'].astype(str).str.strip()

train_df = data[data.par_id_str.isin(train_ids.par_id.astype(str).str.strip())].copy()
dev_df = data[data.par_id_str.isin(dev_ids.par_id.astype(str).str.strip())].copy()

print(train_df.label.value_counts())
print(dev_df.label.value_counts())

In [ ]:
# After filtering, handle the null values immediately
train_df = train_df.dropna(subset=['text']).copy()
dev_df = dev_df.dropna(subset=['text']).copy()

# Forcing text to be of string type - the core to resolving TypeError
train_df['text'] = train_df['text'].astype(str)
dev_df['text'] = dev_df['text'].astype(str)

print(f"Final Train size: {len(train_df)}")
print(f"Final Dev size: {len(dev_df)}")

In [ ]:
# =========================
# EDA 1: Class Distribution
# =========================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x=train_df["label"])
plt.title("Class Distribution in Training Set (Task 1)")
plt.xlabel("Label (0 = Non-PCL, 1 = PCL)")
plt.ylabel("Number of Samples")
plt.show()

print(train_df["label"].value_counts())

In [ ]:
# =========================
# EDA 2: Token Length Distribution
# =========================

from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

lengths = train_df["text"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

plt.figure()
plt.hist(lengths, bins=50)
plt.title("Token Length Distribution (Training Set)")
plt.xlabel("Number of Tokens")
plt.ylabel("Frequency")
plt.show()

print("Mean length:", lengths.mean())
print("Max length:", lengths.max())

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    # Remove the redundant line breaks and Spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Application cleaning
train_df['text'] = train_df['text'].apply(clean_text)
dev_df['text'] = dev_df['text'].apply(clean_text)

**tokenization**

In [ ]:
from transformers import RobertaTokenizer
from datasets import Dataset

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

train_dataset = Dataset.from_pandas(train_df[["text","label"]])
dev_dataset = Dataset.from_pandas(dev_df[["text","label"]])

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
dev_dataset = dev_dataset.rename_column("label", "labels")
train_dataset.set_format("torch", columns=["input_ids","attention_mask","labels"])
dev_dataset.set_format("torch", columns=["input_ids","attention_mask","labels"])

**Calculate class weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df.label),
    y=train_df.label
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

**Custom Trainer**

In [ ]:
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
from torch import nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

**Training Parameters**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
)

**F1 calculation**

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1": f1_score(labels, preds)}

**Training**

In [ ]:
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

trainer.train()

**Dev result**

In [ ]:
metrics = trainer.evaluate()
print(metrics)

**Threshold tuning**

In [ ]:
predictions = trainer.predict(dev_dataset)
logits = predictions.predictions
labels = predictions.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

best_f1 = 0
best_t = 0

for t in np.arange(0.1,0.9,0.05):
    preds = (probs[:,1] > t).astype(int)
    f1 = f1_score(labels, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

In [ ]:
# ==========================================
# Task 1: Generate Test Predictions (test.txt)
# ==========================================
import torch
import numpy as np
import pandas as pd
from datasets import Dataset

print("Starting prediction on Task 4 Test Set...")

# 1. Load the test set
test_path = "task4_test.tsv"
test_df_raw = pd.read_csv(test_path, sep="\t", header=None)
test_df_raw.columns = ["par_id", "art_id", "keyword", "country", "text"]

# 2. Handle null values
test_df_raw['text'] = test_df_raw['text'].fillna("").astype(str)

# 3. turn to HuggingFace Dataset
test_dataset = Dataset.from_pandas(test_df_raw[["text"]])

# 4. Tokenize
def tokenize_for_test(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

test_dataset_mapped = test_dataset.map(tokenize_for_test, batched=True)
test_dataset_mapped.set_format("torch", columns=["input_ids", "attention_mask"])

# 5. Obtain the predicted probability
test_predictions = trainer.predict(test_dataset_mapped)
test_logits = test_predictions.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

# 6. Apply the best_t obtained during training
final_test_preds = (test_probs[:, 1] > best_t).astype(int)

# 7. Save it as test.txt
with open("test.txt", "w", encoding="utf-8") as f:
    for val in final_test_preds:
        f.write(f"{val}\n")

print("-" * 30)
print(f"Success! 'test.txt' has been generated.")
print(f"Total lines: {len(final_test_preds)}")

In [ ]:
# ==========================================
# Task 1: Generate Dev Predictions (dev.txt)
# ==========================================
print("Generating dev.txt for submission...")

# Obtain the predictions of the validation set
dev_predictions = trainer.predict(dev_dataset)
dev_logits = dev_predictions.predictions
dev_probs = torch.softmax(torch.tensor(dev_logits), dim=1).numpy()

# Use the best_t found previously
final_dev_preds = (dev_probs[:, 1] > best_t).astype(int)

# Save file
with open("dev.txt", "w", encoding="utf-8") as f:
    for val in final_dev_preds:
        f.write(f"{val}\n")

print(f"Success! 'dev.txt' generated with {len(final_dev_preds)} lines.")

# Task 2

**Create Task 2 Official Multi-Label Split**

In [ ]:
# =========================
# TASK 2 – Construct Official Multi-Label Split (FIXED)
# =========================

import pandas as pd
from ast import literal_eval

# 1. Read the official split
train_ids = pd.read_csv("train_semeval_parids-labels.csv")
dev_ids = pd.read_csv("dev_semeval_parids-labels.csv")

# 2. Convert the label string to a list (Task 2 is multi-label)
train_ids["label"] = train_ids["label"].apply(literal_eval)
dev_ids["label"] = dev_ids["label"].apply(literal_eval)

# 3. Prepare the basic data
data["par_id_str"] = data["par_id"].astype(str).str.strip()

# 4. Construct train_df_task2
# When merging, only take the text and par_id_str of data to avoid label column conflicts
train_df_task2 = train_ids[["par_id", "label"]].copy()
train_df_task2["par_id_str"] = train_df_task2["par_id"].astype(str).str.strip()

train_df_task2 = train_df_task2.merge(
    data[["par_id_str", "text"]],
    on="par_id_str",
    how="left"
)

# 5. Construct dev_df_task2
dev_df_task2 = dev_ids[["par_id", "label"]].copy()
dev_df_task2["par_id_str"] = dev_df_task2["par_id"].astype(str).str.strip()

dev_df_task2 = dev_df_task2.merge(
    data[["par_id_str", "text"]],
    on="par_id_str",
    how="left"
)

# 6. Clear null values and force type conversion
train_df_task2 = train_df_task2.dropna(subset=["text"]).copy()
dev_df_task2 = dev_df_task2.dropna(subset=["text"]).copy()
train_df_task2["text"] = train_df_task2["text"].astype(str)
dev_df_task2["text"] = dev_df_task2["text"].astype(str)

print("Train size:", len(train_df_task2))
print("Dev size:", len(dev_df_task2))
print("Columns:", train_df_task2.columns.tolist())

In [ ]:
# =========================
# EDA: Task 2 Label Distribution
# =========================

import numpy as np
import matplotlib.pyplot as plt

labels_matrix = np.array(train_df_task2["label"].tolist())

label_freq = labels_matrix.sum(axis=0)

plt.figure()
plt.bar(range(7), label_freq)
plt.title("Task 2 Label Frequency Distribution (Training Set)")
plt.xlabel("Label Index")
plt.ylabel("Positive Sample Count")
plt.show()

print("Label counts:", label_freq)

In [ ]:
# Executed before constructing the Dataset
train_df_task2["label"] = train_df_task2["label"].apply(lambda x: [float(i) for i in x])
dev_df_task2["label"] = dev_df_task2["label"].apply(lambda x: [float(i) for i in x])

**Convert to the HuggingFace Dataset**

In [ ]:
# =========================
# Convert to HuggingFace Dataset
# =========================

from datasets import Dataset

train_dataset_task2 = Dataset.from_pandas(
    train_df_task2[["text","label"]]
)

dev_dataset_task2 = Dataset.from_pandas(
    dev_df_task2[["text","label"]]
)

# The tokenizer uses the previous roberta-base tokenizer

def tokenize_task2(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset_task2 = train_dataset_task2.map(tokenize_task2, batched=True)
dev_dataset_task2 = dev_dataset_task2.map(tokenize_task2, batched=True)

train_dataset_task2 = train_dataset_task2.rename_column("label", "labels")
dev_dataset_task2 = dev_dataset_task2.rename_column("label", "labels")
train_dataset_task2.set_format(
    "torch",
    columns=["input_ids","attention_mask","labels"]
)

dev_dataset_task2.set_format(
    "torch",
    columns=["input_ids","attention_mask","labels"]
)

In [ ]:
# =========================
# Calculate pos_weight for multi-label
# =========================

import torch
import numpy as np

labels_matrix = np.array(train_df_task2["label"].tolist())

# The number of positive samples for each label
pos_counts = labels_matrix.sum(axis=0)

# Negative sample size
neg_counts = len(labels_matrix) - pos_counts

pos_weight = torch.tensor(
    neg_counts / (pos_counts + 1e-6),
    dtype=torch.float32
)

print("Pos weight:", pos_weight)

In [ ]:
# =========================
# Custom Multi-Label Trainer
# =========================

from transformers import RobertaForSequenceClassification
from torch import nn
from transformers import Trainer

class MultiLabelTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Make sure the pos_weight is on the correct device and with the right precision
        device = logits.device
        dtype = logits.dtype
        pw = self.pos_weight.to(device=device, dtype=dtype)

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pw)

        # Explicitly convert labels to the same float type as logits for calculation
        loss = loss_fct(logits, labels.to(dtype))

        return (loss, outputs) if return_outputs else loss

**Training Parameters**

In [ ]:
from transformers import TrainingArguments

training_args_task2 = TrainingArguments(
    output_dir="./task2_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True
)

**Macro F1 calculation**

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics_task2(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    # Make sure that labels are also of integer type
    labels = labels.astype(int)

    return {
        "f1": f1_score(labels, preds, average="macro")
    }

**Training**

In [ ]:
model_task2 = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=7,
    problem_type="multi_label_classification"
)

trainer_task2 = MultiLabelTrainer(
    model=model_task2,
    args=training_args_task2,
    train_dataset=train_dataset_task2,
    eval_dataset=dev_dataset_task2,
    compute_metrics=compute_metrics_task2,
    pos_weight=pos_weight
)

trainer_task2.train()

**Dev Evaluation + Threshold Tuning**

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
import torch

# 1. Obtain the predicted probability
predictions = trainer_task2.predict(dev_dataset_task2)
logits = predictions.predictions
labels = predictions.label_ids
# Use sigmoid to convert logits to probabilities between 0 and 1
probs = torch.sigmoid(torch.tensor(logits)).numpy()

# 2. Find the optimal thresholds for each of the seven categories respectively
best_thresholds = []
per_class_f1 = []

# Task 2 has a total of 7 tags
for i in range(7):
    best_f1_i = 0
    best_t_i = 0.5
    # Search the threshold within a more detailed range (0.01 to 0.99)
    for t in np.arange(0.01, 0.99, 0.01):
        preds_i = (probs[:, i] > t).astype(int)
        # Calculate the binary classification F1 of this category
        f1_i = f1_score(labels[:, i], preds_i, zero_division=0)
        if f1_i > best_f1_i:
            best_f1_i = f1_i
            best_t_i = t

    best_thresholds.append(best_t_i)
    per_class_f1.append(best_f1_i)
    print(f"Label {i} | Best Threshold: {best_t_i:.2f} | F1: {best_f1_i:.4f}")

# 3. Calculate the final Macro F1
final_macro_f1 = np.mean(per_class_f1)
print("-" * 30)
print(f"Average Macro F1 (Per-class Optimized): {final_macro_f1:.4f}")

tmp: Standby for testing

In [ ]:
# ==========================================
# Task 2: Fixed Best Thresholds (From Dev Search)
# ==========================================
best_thresholds = [0.77, 0.66, 0.50, 0.71, 0.95, 0.79, 0.80]

print("Fixed thresholds for Task 2:", best_thresholds)

In [ ]:
# ==========================================
# Task 2: Prepare Test Dataset
# ==========================================
test_path = "task4_test.tsv"
test_df_task2_raw = pd.read_csv(test_path, sep="\t", header=None)
test_df_task2_raw.columns = ["par_id", "art_id", "keyword", "country", "text"]

test_df_task2_raw['text'] = test_df_task2_raw['text'].fillna("").astype(str)
test_df_task2_raw['text'] = test_df_task2_raw['text'].apply(clean_text)

#turn to HuggingFace Dataset
from datasets import Dataset
test_dataset_task2 = Dataset.from_pandas(test_df_task2_raw[["text"]])

# Tokenize
test_dataset_task2 = test_dataset_task2.map(tokenize_task2, batched=True)
test_dataset_task2.set_format("torch", columns=["input_ids", "attention_mask"])

print(f"Test dataset for Task 2 is ready. Size: {len(test_dataset_task2)}")

test_output_task2 = trainer_task2.predict(test_dataset_task2)
test_probs_task2 = torch.sigmoid(torch.tensor(test_output_task2.predictions)).numpy()

# 2. Apply the optimal threshold for classification
# best_thresholds = [0.73, 0.90, 0.59, 0.85, 0.27, 0.67, 0.56]
final_task2_preds = []

for i in range(len(test_probs_task2)):
    row_preds = []
    for j in range(7):
        # Use the corresponding threshold for each tag
        label_pred = 1 if test_probs_task2[i, j] > best_thresholds[j] else 0
        row_preds.append(label_pred)
    final_task2_preds.append(row_preds)

**Special for writing reports**

In [ ]:
# Find the index where Label 6 in the validation set predicted wrongly
# Suppose labels are the true labels of the validation set and probs are the predicted probabilities of the validation set
val_preds_label6 = (probs[:, 6] > best_thresholds[6]).astype(int)
errors_idx = np.where(val_preds_label6 != labels[:, 6])[0]

#Print the text of the first three error samples
print("Label 6 Error Samples Analysis:")
for idx in errors_idx[:3]:
    print(f"Text: {dev_df_task2.iloc[idx]['text']}")
    print(f"True: {labels[idx, 6]}, Pred: {val_preds_label6[idx]}")
    print("-" * 10)

In [ ]:
# Task 1 Error Analysis: Identify False Positives (False positives: Not PCL but predicted as PCL)
# This is very common in PCL tasks
dev_preds_binary = (dev_probs[:, 1] > best_t).astype(int)
binary_labels = dev_df['label'].values

# Find the index of FP
fp_indices = np.where((dev_preds_binary == 1) & (binary_labels == 0))[0]

print("\n--- Task 1 False Positive Analysis (Top 3) ---")
for idx in fp_indices[:3]:
    print(f"Text: {dev_df.iloc[idx]['text']}")
    print(f"Reason: Model incorrectly predicted PCL.")
    print("-" * 10)

In [ ]:
import matplotlib.pyplot as plt

labels_names = ["Unbalanced", "Shallow", "Presupposition", "Authority", "Compassion", "Charity", "Legal"] # 示例名称
plt.bar(range(7), per_class_f1)
plt.xticks(range(7), labels_names, rotation=45)
plt.ylabel('F1 Score')
plt.title('Task 2 Per-class Performance')
for i, v in enumerate(per_class_f1):
    plt.text(i - 0.2, v + 0.01, f"{v:.2f}", fontsize=9)
plt.show()

In [ ]:
import os

# Create the BestModel folder
os.makedirs("BestModel", exist_ok=True)

# Save the model weights and configuration files
# This way, GTA will know which checkpoint you are using
trainer.save_model("./BestModel/saved_model")
tokenizer.save_pretrained("./BestModel/saved_model")

print("Best model weights and tokenizer saved to ./BestModel/saved_model")